# ESAM-E Teacher - Export 3D Point Cloud (.ply)

Runs the online teacher on SceneNN 322 to generate `teacher_val.ply`.
Runtime: ~5-7 minutes on T4 GPU.

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/tools/distill /kaggle/working/logs /kaggle/working/distill

cat > /kaggle/working/tools/distill/config.py <<'PYEOF_CONFIG'
"""Paths, data parameters and training parameters of the distillation study.

Every value can be overridden with an environment variable, so the experiment
scripts never need command line parsing.  The experiments themselves live in
`experiments/<name>/settings.json`.
"""
import os
from pathlib import Path

# ============================================================
# 1. LOCATIONS
# ============================================================
WORK = Path(os.environ.get("ESAM_WORK", "/kaggle/working"))
ESAM_ROOT = Path(os.environ.get("ESAM_ROOT", WORK / "ESAM"))
DATA_ROOT = Path(os.environ.get("ESAM_DATA", WORK / "data/scenenn_stream/322"))
RAW_ROOT = Path(os.environ.get("ESAM_RAW", WORK / "tmp/scenenn"))
FRAMES = Path(os.environ.get("ESAM_FRAMES", WORK / "distill/frames"))
TEACHER = Path(os.environ.get("ESAM_TEACHER", WORK / "distill/teacher"))
STUDENT = Path(os.environ.get("ESAM_STUDENT", WORK / "distill/student"))

# ============================================================
# 2. MODEL FILES
# ============================================================
ESAM_CKPT = Path(os.environ.get(
    "ESAM_CKPT", ESAM_ROOT / "work_dirs/ckpt/ESAM-E_CA_online_epoch_128.pth"))
TEACHER_CFG = Path(os.environ.get(
    "ESAM_TEACHER_CFG", ESAM_ROOT / "configs/ESAM-E_CA/ESAM-E_online_stream.py"))
STUDENT_CFG = Path(os.environ.get(
    "ESAM_STUDENT_CFG", WORK / "tools/esampp_tiny_scenenn.py"))

# ============================================================
# 3. DATA PARAMETERS
# ============================================================
NUM_FRAMES = int(os.environ.get("ESAM_NUM_FRAMES", 80))
VAL_START = int(os.environ.get("ESAM_VAL_START", 40))
CLUSTERS = int(os.environ.get("ESAM_CLUSTERS", 100))
SCORE_THR = float(os.environ.get("ESAM_SCORE_THR", 0.3))
MIN_POINTS = int(os.environ.get("ESAM_MIN_POINTS", 100))

# ============================================================
# 4. TRAINING PARAMETERS
# ============================================================
MAX_MINUTES = float(os.environ.get("ESAM_MAX_MINUTES", 110))
LR = float(os.environ.get("ESAM_LR", 1e-3))
DEVICE = "cuda"

# ============================================================
# 5. EXPERIMENT SWITCHES
# ============================================================
FREEZE_SHARED = os.environ.get("ESAM_FREEZE_SHARED", "0") == "1"   # train only the TinyExpert
TARGET = os.environ.get("ESAM_TARGET", "teacher")                  # "teacher" or "gt"
PYEOF_CONFIG

cat > /kaggle/working/tools/distill/common.py <<'PYEOF_COMMON'
"""Geometry, colour, PLY and evaluation helpers shared by the experiment scripts."""
import json
import os

import numpy as np

# ============================================================
# 1. GEOMETRY
# ============================================================
# SceneNN is Y-up, the models expect Z-up: R_x(+90) maps (x, y, z) -> (x, -z, y).
R_ZUP = np.array([[1.0, 0.0, 0.0],
                  [0.0, 0.0, -1.0],
                  [0.0, 1.0, 0.0]])


def zup(points):
    """Rotate SceneNN world points into the Z-up frame of the models."""
    return (np.asarray(points, dtype=np.float64) @ R_ZUP.T).astype(np.float32)


# ============================================================
# 2. COLOURS
# ============================================================
def palette(seed=0, n=256):
    """Deterministic instance colours."""
    rng = np.random.default_rng(seed)
    colors = rng.integers(0, 256, size=(n, 3), dtype=np.uint8)
    colors[-1] = (190, 190, 190)
    return colors


def mask_to_colors(masks, scores=None, colors=None):
    """One stable colour per mask row.  Row order is preserved."""
    masks = np.asarray(masks, dtype=bool)
    colors = palette() if colors is None else colors
    index = np.full(masks.shape[1], -1, dtype=np.int64)
    order = np.argsort(np.asarray(scores)) if scores is not None else range(len(masks))
    for row in order:
        index[masks[row]] = row
    return colors[index]


# ============================================================
# 3. PLY AND POINT CLOUD IO
# ============================================================
def write_ply(path, xyz, rgb):
    """Write a binary little-endian PLY."""
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    header = ("ply\n"
              "format binary_little_endian 1.0\n"
              f"element vertex {len(xyz)}\n"
              "property float x\nproperty float y\nproperty float z\n"
              "property uchar red\nproperty uchar green\nproperty uchar blue\n"
              "end_header\n")
    record = np.empty(len(xyz), dtype=[("x", "<f4"), ("y", "<f4"), ("z", "<f4"),
                                       ("red", "u1"), ("green", "u1"), ("blue", "u1")])
    record["x"], record["y"], record["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    record["red"], record["green"], record["blue"] = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        record.tofile(f)


def voxel_downsample(xyz, rgb, voxel=0.02):
    """Keep one point per occupied voxel."""
    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    if len(xyz) == 0:
        return xyz, rgb
    keys = np.floor((xyz - xyz.min(axis=0)) / voxel).astype(np.int64)
    keys = np.clip(keys, 0, (1 << 21) - 1)
    packed = (keys[:, 0] << 42) | (keys[:, 1] << 21) | keys[:, 2]
    keep = np.sort(np.unique(packed, return_index=True)[1])
    return xyz[keep], rgb[keep]


def save_preview(path, xyz, rgb, title=""):
    """Save a two-view matplotlib preview of a point cloud."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    xyz = np.asarray(xyz, dtype=np.float32)
    rgb = np.asarray(rgb, dtype=np.uint8)
    if len(xyz) > 90000:
        sel = np.random.default_rng(0).choice(len(xyz), 90000, replace=False)
        xyz, rgb = xyz[sel], rgb[sel]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), dpi=140)
    for ax, (name, a, b) in zip(axes, [("X / Z", 0, 2), ("X / Y", 0, 1)]):
        ax.scatter(xyz[:, a], xyz[:, b], c=rgb / 255.0, s=0.25, linewidths=0)
        ax.set_title(name)
        ax.set_aspect("equal", adjustable="box")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 4. EVALUATION
# ============================================================
def iou_matrix(pred, gt):
    """IoU between predicted masks [P, N] and ground truth masks [G, N]."""
    pred = np.asarray(pred, dtype=np.float32)
    gt = np.asarray(gt, dtype=np.float32)
    if pred.size == 0 or gt.size == 0:
        return np.zeros((len(pred), len(gt)), dtype=np.float32)
    inter = pred @ gt.T
    union = pred.sum(1)[:, None] + gt.sum(1)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-6), 0.0)


def average_precision(pred_masks, pred_scores, gt_masks, threshold=0.5):
    """Class agnostic AP at one IoU threshold.  gt_masks: [G, N] bool."""
    if len(gt_masks) == 0 or len(pred_masks) == 0:
        return 0.0
    ious = iou_matrix(pred_masks, gt_masks)
    order = np.argsort(-np.asarray(pred_scores))
    used = np.zeros(len(gt_masks), dtype=bool)
    tp = np.zeros(len(order), dtype=np.float32)
    for rank, row in enumerate(order):
        best = int(np.argmax(ious[row]))
        if ious[row, best] >= threshold and not used[best]:
            used[best] = True
            tp[rank] = 1.0
    if tp.sum() == 0:
        return 0.0
    tp_c = np.cumsum(tp)
    fp_c = np.cumsum(1.0 - tp)
    recall = tp_c / len(gt_masks)
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-6)
    recall = np.concatenate([[0.0], recall, [1.0]])
    precision = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    idx = np.where(recall[1:] != recall[:-1])[0]
    return float(np.sum((recall[idx + 1] - recall[idx]) * precision[idx + 1]))


def instance_ap(pred_masks, pred_scores, gt_ins, min_gt_points=50):
    """AP averaged over IoU 0.50:0.95, plus AP50 and AP25, for one frame."""
    pred_masks = np.asarray(pred_masks, dtype=bool)
    pred_scores = np.asarray(pred_scores, dtype=np.float32)
    gt_ins = np.asarray(gt_ins, dtype=np.int64)
    if len(pred_masks) > 300:
        top = np.argsort(-pred_scores)[:300]
        pred_masks, pred_scores = pred_masks[top], pred_scores[top]
    gt_ids = [x for x in np.unique(gt_ins) if x > 0]
    gt_masks = [gt_ins == gid for gid in gt_ids
                if (gt_ins == gid).sum() >= min_gt_points]
    if not gt_masks:
        return {"AP": 0.0, "AP50": 0.0, "AP25": 0.0,
                "num_pred": int(len(pred_masks)), "num_gt": 0}
    gt_masks = np.stack(gt_masks)
    thresholds = list(np.arange(0.50, 0.96, 0.05))
    values = [average_precision(pred_masks, pred_scores, gt_masks, t) for t in thresholds]
    return {"AP": float(np.mean(values)),
            "AP50": float(average_precision(pred_masks, pred_scores, gt_masks, 0.50)),
            "AP25": float(average_precision(pred_masks, pred_scores, gt_masks, 0.25)),
            "num_pred": int(len(pred_masks)), "num_gt": int(len(gt_masks))}


# ============================================================
# 5. JSON IO
# ============================================================
def write_json(path, data):
    """Write a JSON report file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=float)
PYEOF_COMMON

cat > /kaggle/working/tools/distill/data.py <<'PYEOF_DATA'
"""Build the SceneNN 322 frames used by every experiment.

Each frame holds the 20 000 points of the scene in world Z-up coordinates, the
super point grouping and the exact per-point instance ids.
"""
import numpy as np
from sklearn.cluster import KMeans

import config as C
from common import write_json, zup

# ImageNet statistics used by the ESAM models for the colour channels.
RGB_MEAN = np.array([0.47793125906962, 0.4303257521323044, 0.3749598901421883])
RGB_STD = np.array([0.2294476613552524, 0.22136130327957347, 0.22412925035485958])


# ============================================================
# 1. FRAME LOADING
# ============================================================
def load_frame(frame_id):
    """Load one SceneNN frame: points, super points, instance ids, colours."""
    name = f"{frame_id:05d}.npy"
    scene = C.RAW_ROOT / "oni" / "322"
    point = np.load(scene / "point" / name)
    ins = np.load(scene / "ins" / name)
    pose = np.load(scene / "pose" / name)

    xyz = point[:, :3]
    xyz = (pose @ np.c_[xyz, np.ones(len(xyz))].T).T[:, :3]
    xyz = zup(xyz)

    rgb = point[:, 3:6].astype(np.float32)
    norm_rgb = (rgb / 255.0 - RGB_MEAN) / RGB_STD
    super_points = KMeans(n_clusters=C.CLUSTERS, n_init=10,
                           random_state=0).fit_predict(xyz)
    return (np.c_[xyz, norm_rgb].astype(np.float32),
            super_points.astype(np.int64),
            ins.astype(np.int64),
            rgb.astype(np.uint8))


def frame_ids():
    """Frame ids of the study: the first NUM_FRAMES ids of the converted stream."""
    ids = [int(x) for x in (C.DATA_ROOT / "frame_ids.txt").read_text().split()]
    return ids[:C.NUM_FRAMES]


# ============================================================
# 2. MAIN
# ============================================================
def main():
    """Write one npz per frame plus a small meta file."""
    C.FRAMES.mkdir(parents=True, exist_ok=True)
    ids = frame_ids()
    for index, frame_id in enumerate(ids):
        points, super_points, ins, rgb = load_frame(frame_id)
        np.savez_compressed(C.FRAMES / f"frame_{index:04d}.npz",
                            points=points, sp_pts_mask=super_points,
                            gt_ins=ins, rgb_raw=rgb,
                            frame_id=np.array([frame_id]))
        if (index + 1) % 20 == 0:
            print(f"prepared {index + 1}/{len(ids)} frames")
    write_json(C.FRAMES / "meta.json",
               {"frames": len(ids), "frame_ids": ids,
                "points_per_frame": 20000, "clusters": C.CLUSTERS,
                "rotation": "R_x(+90) Y-up -> Z-up"})
    print("frames ready:", C.FRAMES)


if __name__ == "__main__":
    main()
PYEOF_DATA

cat > /kaggle/working/tools/distill/teacher.py <<'PYEOF_TEACHER'
"""Run the ESAM-E online teacher over the prepared frames.

The teacher produces the pseudo instances that the student imitates, and the
reference AP that every experiment is measured against.
"""
import glob
import sys
import time

import numpy as np
import torch

import config as C
from common import (instance_ap, mask_to_colors, palette, save_preview,
                    voxel_downsample, write_json, write_ply)

sys.path.insert(0, str(C.ESAM_ROOT))


# ============================================================
# 1. TEACHER MODEL
# ============================================================
def build_teacher():
    """Build ESAM-E with the published CA online checkpoint."""
    import oneformer3d  # noqa: F401
    from mmengine.config import Config
    from mmengine.registry import init_default_scope
    from mmengine.runner import load_checkpoint
    from mmdet3d.registry import MODELS

    cfg = Config.fromfile(C.TEACHER_CFG)
    init_default_scope(cfg.get("default_scope", "mmdet3d"))
    model = MODELS.build(cfg.model)
    load_checkpoint(model, str(C.ESAM_CKPT), map_location="cpu")
    model.cfg = cfg
    model.map_to_rec_pcd = False
    return model.to(C.DEVICE).eval()


def teacher_frame(model, frame):
    """Predict the instances of one frame with the online ESAM-E model."""
    from mmengine.dataset import pseudo_collate
    from mmdet3d.structures import Det3DDataSample, PointData

    points = torch.from_numpy(frame["points"]).float().to(C.DEVICE)
    sample = Det3DDataSample()
    gt_pts = PointData()
    gt_pts["sp_pts_mask"] = torch.from_numpy(frame["sp_pts_mask"]).long().to(C.DEVICE)
    sample.gt_pts_seg = gt_pts
    data = pseudo_collate([dict(inputs={"points": points}, data_samples=sample)])
    with torch.no_grad():
        result = model.test_step(data)
    pred = result[0].pred_pts_seg
    return (np.asarray(pred.pts_instance_mask[0], dtype=bool),
            np.asarray(pred.instance_scores, dtype=np.float32))


# ============================================================
# 2. MAIN
# ============================================================
def main():
    """Predict all frames, store the pseudo instances and write the metrics."""
    files = sorted(glob.glob(str(C.FRAMES / "frame_*.npz")))
    C.TEACHER.mkdir(parents=True, exist_ok=True)
    colors = palette()
    model = build_teacher()

    points_all, gt_all, counts = [], [], []
    frame_ap, stats = [], []
    total = 0
    start = time.time()

    for index, path in enumerate(files):
        data = np.load(path)
        points = data["points"].astype(np.float32)
        gt = data["gt_ins"].astype(np.int64)
        num_points = len(points)

        merged, scores = teacher_frame(model, {k: data[k] for k in data.files})
        current = merged[:, total:total + num_points]
        frame_ap.append(instance_ap(current, scores, gt))

        keep = [i for i in range(len(current))
                if current[i].sum() >= C.MIN_POINTS and scores[i] >= C.SCORE_THR]
        masks = current[keep] if keep else np.zeros((0, num_points), dtype=bool)
        mask_scores = scores[keep] if keep else np.zeros((0,), dtype=np.float32)
        np.savez_compressed(C.TEACHER / f"pseudo_{index:04d}.npz",
                            masks=masks, scores=mask_scores,
                            frame_id=data["frame_id"])

        points_all.append(points[:, :3])
        gt_all.append(gt)
        counts.append(len(masks))
        stats.append({"index": index, "frame_id": int(data["frame_id"][0]),
                      "instances": int(len(masks)),
                      "max_mask_points": int(masks.sum(axis=1).max()) if len(masks) else 0})
        total += num_points

        if index < 4:
            save_preview(C.TEACHER / f"teacher_preview_{index:04d}.png",
                         points[:, :3], mask_to_colors(masks, mask_scores, colors),
                         title=f"ESAM-E teacher frame {index} ({len(masks)} instances)")
        if (index + 1) % 20 == 0:
            print(f"teacher frame {index + 1}/{len(files)}")

    np.savez_compressed(C.TEACHER / "teacher_final.npz",
                        points=np.concatenate(points_all),
                        gt_ins=np.concatenate(gt_all),
                        instances=np.array(counts))

    val_pts = np.concatenate(points_all[C.VAL_START:])
    val_offset = sum(len(p) for p in points_all[:C.VAL_START])
    val_masks = merged[:, val_offset:]
    val_colors = mask_to_colors(val_masks, scores, colors)
    write_ply(C.TEACHER / "teacher_val.ply", val_pts, val_colors)
    down_xyz, down_colors = voxel_downsample(val_pts, val_colors, 0.02)
    write_ply(C.TEACHER / "teacher_val_voxel02.ply", down_xyz, down_colors)

    # ============================================================
    # 3. METRICS AND SANITY GATE
    # ============================================================
    def mean_ap(selection):
        chosen = [a for i, a in enumerate(frame_ap)
                  if selection(i) and a["num_gt"] > 0]
        if not chosen:
            return {"AP": 0.0, "AP50": 0.0, "AP25": 0.0, "frames": 0}
        return {"AP": float(np.mean([a["AP"] for a in chosen])),
                "AP50": float(np.mean([a["AP50"] for a in chosen])),
                "AP25": float(np.mean([a["AP25"] for a in chosen])),
                "frames": len(chosen)}

    ap_val = mean_ap(lambda i: i >= C.VAL_START)
    ap_all = mean_ap(lambda i: True)
    fractions = [s["max_mask_points"] / 20000.0 for s in stats if s["max_mask_points"]]
    gate = {"ok": bool(len(files) >= 20 and np.median(counts) >= 2
                       and 0.01 <= np.median(fractions) <= 0.95),
            "frames": len(files),
            "median_instances": float(np.median(counts)),
            "median_max_mask_fraction": float(np.median(fractions))}
    write_json(C.TEACHER / "teacher_metrics.json",
               {"frames": len(files), "elapsed_sec": time.time() - start,
                "teacher_ap_val": ap_val, "teacher_ap_all": ap_all,
                "gate": gate, "frame_stats": stats})
    print("teacher AP (val):", ap_val)
    print("gate:", gate)


if __name__ == "__main__":
    main()
PYEOF_TEACHER

In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
SRC=/kaggle/input/notebooks/abdesselamaimen01/notebook8491230ab9
mkdir -p "$W/logs" "$W/.state" "$W/tools"
echo "restoring saved environment"
tar -xzf "$SRC/esam_env.tar.gz" -C "$W" --ignore-zeros \
  2>&1 | tee "$W/logs/00_restore.log" || true
for d in ESAM FastSAM data; do
  rm -rf "$W/$d"
  cp -a "$SRC/$d" "$W/"
done
cp "$SRC/setup.sh" "$W/setup.sh" 2>/dev/null || true
export PATH="$W/mc/envs/ESAM/bin:$PATH"
export LD_LIBRARY_PATH="$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}"
export PYTHONPATH="$W/ESAM:$W/FastSAM"
df -h "$W" | tail -1


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
export PATH=$W/mc/envs/ESAM/bin:$PATH
export LD_LIBRARY_PATH=$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}
export CUDA_VISIBLE_DEVICES=0
cd $W
$W/mc/envs/ESAM/bin/python $W/tools/distill/data.py 2>&1 | tee $W/logs/20_prepare.log
ls -lh $W/distill/frames | tail -3


In [ ]:
%%bash
set -euo pipefail
W=/kaggle/working
export PATH=$W/mc/envs/ESAM/bin:$PATH
export LD_LIBRARY_PATH=$W/mc/envs/ESAM/lib:$W/mc/envs/ESAM/lib64:${LD_LIBRARY_PATH:-}
export CUDA_VISIBLE_DEVICES=0
cd $W/ESAM
$W/mc/envs/ESAM/bin/python $W/tools/distill/teacher.py 2>&1 | tee $W/logs/30_teacher.log
cat $W/distill/teacher/teacher_metrics.json | tail -40


In [ ]:
%%bash
set +e
W=/kaggle/working
echo "================ OUTPUT POINT CLOUDS ================"
ls -lh $W/distill/teacher/*.ply
